# Partie 2 — Classification d’images Fashion-MNIST

Reprise des étapes du tutoriel : chargement, préparation, réseau convolutif, entraînement, évaluation, graphiques et sauvegarde.

**Prérequis :** placer `fashion-mnist_train.csv` et `fashion-mnist_test.csv` dans `analyse_image/datas/` (depuis Moodle). Ces gros fichiers CSV **ne doivent pas être commités**.

**Exécution :** ouvrir ce notebook dans VS Code avec le noyau Python contenant `tensorflow`, `keras`, `pandas`, `numpy`, `matplotlib`, `scikit-learn` puis utiliser **Run All**. Le modèle initial utilise **10 epochs**, conformément au tutoriel. Les entraînements supplémentaires de **50 epochs** et le réseau complexe sont optionnels et désactivés par défaut.

In [ ]:
from pathlib import Path
import pandas as pnd
import numpy as np
from matplotlib import pyplot as plt
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow import keras

# Tutoriel : entraînement sur processeur pour éviter les problèmes de GPU
try:
    tf.config.set_visible_devices([], 'GPU')
except RuntimeError:
    pass

LONGUEUR_IMAGE = 28
LARGEUR_IMAGE = 28
RACINE = Path.cwd()
if not (RACINE / 'datas' / 'fashion-mnist_train.csv').exists():
    RACINE = Path.cwd() / 'analyse_image'  # cas : notebook lancé depuis la racine Git
assert (RACINE / 'datas' / 'fashion-mnist_train.csv').exists(), 'Téléchargez le CSV train de Moodle dans analyse_image/datas/'
assert (RACINE / 'datas' / 'fashion-mnist_test.csv').exists(), 'Téléchargez le CSV test de Moodle dans analyse_image/datas/'
(RACINE / 'modele').mkdir(exist_ok=True)
print('TensorFlow :', tf.__version__, '| données :', RACINE / 'datas')

## 1. Charger et visualiser la première image

In [ ]:
observations_entrainement = pnd.read_csv(RACINE / 'datas' / 'fashion-mnist_train.csv')
X = np.array(observations_entrainement.iloc[:, 1:])
premiereImage = X[0].reshape([LONGUEUR_IMAGE, LARGEUR_IMAGE])
plt.imshow(premiereImage, cmap='gray')
plt.title('Première image Fashion-MNIST')
plt.axis('off')
plt.show()
print('Observations :', len(observations_entrainement), '| pixels :', X.shape[1])

## 2. Préparer l’apprentissage (80 %) et la validation (20 %)

In [ ]:
y = keras.utils.to_categorical(np.array(observations_entrainement.iloc[:, 0]), num_classes=10)
X_apprentissage, X_validation, y_apprentissage, y_validation = train_test_split(
    X, y, test_size=0.2, random_state=13
)
X_apprentissage = X_apprentissage.reshape(X_apprentissage.shape[0], LARGEUR_IMAGE, LONGUEUR_IMAGE, 1).astype('float32') / 255
X_validation = X_validation.reshape(X_validation.shape[0], LARGEUR_IMAGE, LONGUEUR_IMAGE, 1).astype('float32') / 255

observations_test = pnd.read_csv(RACINE / 'datas' / 'fashion-mnist_test.csv')
X_test = np.array(observations_test.iloc[:, 1:])
y_test = keras.utils.to_categorical(np.array(observations_test.iloc[:, 0]), num_classes=10)
X_test = X_test.reshape(X_test.shape[0], LARGEUR_IMAGE, LONGUEUR_IMAGE, 1).astype('float32') / 255
print('Apprentissage :', X_apprentissage.shape, '| Validation :', X_validation.shape, '| Test :', X_test.shape)

## 3. CNN à une couche de convolution — tutoriel

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Flatten, Input, Conv2D, MaxPooling2D

dimentionImage = (LARGEUR_IMAGE, LONGUEUR_IMAGE, 1)
reseauNeurone1Convolution = Sequential([
    Input(shape=dimentionImage),
    Conv2D(32, kernel_size=(3, 3), activation='relu'),
    MaxPooling2D(pool_size=(2, 2)),
    Dropout(0.2),
    Flatten(),
    Dense(128, activation='relu'),
    Dense(10, activation='softmax')
])
reseauNeurone1Convolution.compile(
    loss=keras.losses.categorical_crossentropy,
    optimizer=tf.keras.optimizers.Adam(),
    metrics=['accuracy']
)
reseauNeurone1Convolution.summary()

## 4. Entraînement initial — 10 epochs

In [ ]:
historique_apprentissage = reseauNeurone1Convolution.fit(
    X_apprentissage, y_apprentissage,
    batch_size=256, epochs=10, verbose=1,
    validation_data=(X_validation, y_validation)
)
evaluation = reseauNeurone1Convolution.evaluate(X_test, y_test, verbose=0)
print('Erreur :', evaluation[0])
print('Précision :', evaluation[1])

## 5. Courbes de précision et d’erreur

In [ ]:
plt.plot(historique_apprentissage.history['accuracy'])
plt.plot(historique_apprentissage.history['val_accuracy'])
plt.title('Précision du modèle')
plt.ylabel('Précision')
plt.xlabel('Epoch')
plt.legend(['Apprentissage', 'Validation'], loc='upper left')
plt.show()

plt.plot(historique_apprentissage.history['loss'])
plt.plot(historique_apprentissage.history['val_loss'])
plt.title('Erreur')
plt.ylabel('Erreur')
plt.xlabel('Epoch')
plt.legend(['Apprentissage', 'Validation'], loc='upper left')
plt.show()

## 6. Sauvegarde du modèle entraîné

Le fichier JSON décrit l’architecture et le fichier H5 contient les poids. `test_model.ipynb` utilisera ces deux fichiers.

In [ ]:
model_json = reseauNeurone1Convolution.to_json()
with open(RACINE / 'modele' / 'modele.json', 'w', encoding='utf-8') as json_file:
    json_file.write(model_json)
reseauNeurone1Convolution.save_weights(str(RACINE / 'modele' / 'modele.weights.h5'))
print('Modèle sauvegardé :', RACINE / 'modele')

## 7. Optionnel — augmentation d’images et entraînement long

Le tutoriel propose ensuite **50 epochs supplémentaires** avec rotation, translations et zoom. Cette étape est coûteuse et **désactivée par défaut** pour permettre une exécution complète rapide. Pour la réaliser, passer la variable à `True`, puis réexécuter la cellule et la cellule de sauvegarde.

In [ ]:
EXECUTER_ENTRAINEMENT_LONG = False
if EXECUTER_ENTRAINEMENT_LONG:
    from tensorflow.keras.preprocessing.image import ImageDataGenerator
    generateur_images = ImageDataGenerator(
        rotation_range=8, width_shift_range=0.08,
        shear_range=0.3, height_shift_range=0.08, zoom_range=0.08
    )
    nouvelles_images_apprentissage = generateur_images.flow(X_apprentissage, y_apprentissage, batch_size=256)
    nouvelles_images_validation = generateur_images.flow(X_validation, y_validation, batch_size=256)
    historique_apprentissage = reseauNeurone1Convolution.fit(
        nouvelles_images_apprentissage,
        steps_per_epoch=max(1, len(X_apprentissage) // 256), epochs=50,
        validation_data=nouvelles_images_validation,
        validation_steps=max(1, len(X_validation) // 256), verbose=1
    )
    evaluation = reseauNeurone1Convolution.evaluate(X_test, y_test, verbose=0)
    print('Après augmentation — Erreur :', evaluation[0], '| Précision :', evaluation[1])
    with open(RACINE / 'modele' / 'modele.json', 'w', encoding='utf-8') as json_file:
        json_file.write(reseauNeurone1Convolution.to_json())
    reseauNeurone1Convolution.save_weights(str(RACINE / 'modele' / 'modele.weights.h5'))
else:
    print('Étape facultative (50 epochs) non exécutée.')

## 8. Optionnel — réseau à quatre convolutions

Ce réseau plus complexe est explicitement facultatif dans le tutoriel et n’est pas entraîné par défaut.

In [ ]:
EXECUTER_RESEAU_OPTIONNEL = False
if EXECUTER_RESEAU_OPTIONNEL:
    from tensorflow.keras.layers import BatchNormalization
    reseauNeurones4Convolution = Sequential([
        Input(shape=dimentionImage),
        Conv2D(32, (3, 3), activation='relu'), BatchNormalization(),
        Conv2D(32, (3, 3), activation='relu'), BatchNormalization(),
        MaxPooling2D((2, 2)), Dropout(0.25),
        Conv2D(64, (3, 3), activation='relu'), BatchNormalization(), Dropout(0.25),
        Conv2D(128, (3, 3), activation='relu'), BatchNormalization(),
        MaxPooling2D((2, 2)), Dropout(0.25),
        Flatten(), Dense(512, activation='relu'), BatchNormalization(), Dropout(0.5),
        Dense(128, activation='relu'), BatchNormalization(), Dropout(0.5),
        Dense(10, activation='softmax')
    ])
    reseauNeurones4Convolution.compile(
        loss=keras.losses.categorical_crossentropy,
        optimizer=tf.keras.optimizers.Adam(), metrics=['accuracy']
    )
    historique_reseau_4 = reseauNeurones4Convolution.fit(
        X_apprentissage, y_apprentissage, batch_size=256, epochs=10, verbose=1,
        validation_data=(X_validation, y_validation)
    )
    print('Réseau 4 convolutions (erreur, précision) :', reseauNeurones4Convolution.evaluate(X_test, y_test, verbose=0))
else:
    print('Réseau complexe facultatif non exécuté.')